# 17 - The guided reading on a third sample of new accounts

The two local LLMs read 255 new accounts (seed 102, the strata of the test sample) alone (K1) and with the guide and four solved examples (K1_guide_shots). This shows whether the gain found on the test sample holds on accounts that played no part in writing the guide.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import binomtest, norm
from sklearn.metrics import roc_auc_score

from skeptical_second_reader import fresh, guide
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.pipeline import READ_ALONE, READ_SHOTS
from skeptical_second_reader.reading_sample import (
    MATCHED_NAME,
    OUTSIDE_NAME,
    RANDOM_NAME,
)
from skeptical_second_reader.second_reader import NUM_CTX, SUSPICIOUS

MODELS = ["gemma4:latest", "qwen3.5:9b-q4_K_M"]
CHOSEN = READ_SHOTS
CONDITIONS = [READ_ALONE, CHOSEN]
CLEAN = [MATCHED_NAME, RANDOM_NAME]
DRAWS = 2000
SEED = 15

reading_dir = DATA_INTERIM / "reading"
pool_dir = reading_dir / fresh.run_name(102, "full")
small_dir = reading_dir / fresh.run_name(101, "small")
samples = {
    "dev": pd.read_parquet(DATA_INTERIM / "reading_dev_cases.parquet")["stratum"],
    "test": pd.read_parquet(DATA_INTERIM / "reading_test_cases.parquet")["stratum"],
    "seed 101 small": pd.read_parquet(small_dir / "cases.parquet")["stratum"],
    "seed 102 pool": pd.read_parquet(pool_dir / "cases.parquet")["stratum"],
}
pool = samples["seed 102 pool"]
truth = {name: fresh.truth_of(s) for name, s in samples.items()}
patterns = sorted(s for s in pool.unique() if s.startswith("pattern "))
ORDER = patterns + [OUTSIDE_NAME] + CLEAN
print(
    f"pool accounts {len(pool)}  positives {truth['seed 102 pool'].sum()}  "
    f"clean {(truth['seed 102 pool'] == 0).sum()}"
)
print(pool.value_counts().reindex(ORDER).to_string())

pool accounts 255  positives 135  clean 120
stratum
pattern BIPARTITE             15
pattern CYCLE                 15
pattern FAN-IN                15
pattern FAN-OUT               15
pattern GATHER-SCATTER        15
pattern RANDOM                15
pattern SCATTER-GATHER        15
pattern STACK                 15
positive, outside the file    15
clean, matched activity       60
clean, random                 60


## 1. Integrity

Checks that the accounts are new, every answer is there and usable, and the prompts carry the same guide and examples as the test sample, so the comparison is fair.

In [2]:
for name, s in samples.items():
    if name != "seed 102 pool":
        shared = len(pool.index.intersection(s.index))
        print(f"accounts shared with {name}: {shared}")
        assert shared == 0
assert (
    pool.value_counts().sort_index().equals(samples["test"].value_counts().sort_index())
)


def prepare(frame):
    frame = frame[frame["condition"].isin(CONDITIONS)].copy()
    frame["usable"] = frame["decision"].notna() & frame["confidence"].notna()
    frame["says"] = frame["decision"] == SUSPICIOUS
    frame["conf_score"] = np.where(
        frame["says"], frame["confidence"], 1 - frame["confidence"]
    )
    return frame


def load_reading(sample):
    alone = pd.read_parquet(reading_dir / f"results_{sample}.parquet")
    guided = pd.read_parquet(reading_dir / f"results_{sample}_guided.parquet")
    return prepare(
        pd.concat([alone[alone["condition"] == READ_ALONE], guided], ignore_index=True)
    )


results = {
    "dev": load_reading("dev"),
    "test": load_reading("test"),
    "seed 101 small": prepare(pd.read_parquet(small_dir / "results.parquet")),
    "seed 102 pool": prepare(pd.read_parquet(pool_dir / "results.parquet")),
}
r, t = results["seed 102 pool"], results["test"]
assert set(r["model"]) == set(MODELS)
assert set(r["condition"]) == set(CONDITIONS)
assert not r.duplicated(["model", "condition", "account"]).any()
for (model, condition), part in r.groupby(["model", "condition"]):
    assert set(part["account"]) == set(pool.index), (model, condition)
assert (r["group"] == pool.loc[r["account"]].to_numpy()).all()
assert (r.groupby("model")["digest"].nunique() == 1).all()
assert (
    r.groupby("model")["digest"].first() == t.groupby("model")["digest"].first()
).all()
assert set(r["think"].astype(str)) == set(t["think"].astype(str))


def shots_part(prompt):
    start = prompt.index("SOLVED EXAMPLES")
    return prompt[start : prompt.index("\n\nACCOUNT A", start)]


def guided_prompts(path):
    p = pd.read_parquet(path)
    return p.loc[p["condition"] == CHOSEN, "prompt"]


pool_prompts = guided_prompts(pool_dir / "prompts.parquet")
test_prompts = guided_prompts(reading_dir / "prompts_test_guided.parquet")
assert len(pool_prompts) == len(pool)
assert pool_prompts.str.contains(guide.GUIDE, regex=False).all()
assert set(pool_prompts.map(shots_part)) == set(test_prompts.map(shots_part))
assert len(set(test_prompts.map(shots_part))) == 1
print("guide and examples: the same in every pool and test prompt")

r["total_tokens"] = r["prompt_tokens"] + r["answer_tokens"]
tokens = r.groupby(["model", "condition"]).agg(
    answers=("account", "size"),
    not_usable=("usable", lambda s: int((~s).sum())),
    max_prompt_tokens=("prompt_tokens", "max"),
    max_prompt_and_answer=("total_tokens", "max"),
    at_window=("prompt_tokens", lambda s: int((s >= NUM_CTX).sum())),
)
print(f"context window {NUM_CTX}")
print(tokens.to_string())

accounts shared with dev: 0
accounts shared with test: 0
accounts shared with seed 101 small: 0


guide and examples: the same in every pool and test prompt
context window 4096
                                  answers  not_usable  max_prompt_tokens  max_prompt_and_answer  at_window
model             condition                                                                               
gemma4:latest     K1                  255           0               1506                   1706          0
                  K1_guide_shots      255           0               2959                   3183          0
qwen3.5:9b-q4_K_M K1                  255           0               1470                   1711          0
                  K1_guide_shots      255           0               2928                   3134          0


## 2. Share flagged per stratum

How often each reader says suspicious in each group: recall on the positives, false alarms on the clean accounts. Shows where the guide changes the decisions.

In [3]:
def wilson(k, n, level=0.95):
    if n == 0:
        return np.nan, np.nan
    z = norm.ppf(0.5 + level / 2)
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


def usable_accounts(frame):
    ok = frame.groupby(["model", "account"])["usable"].all()
    return {m: ok[m][ok[m]].index for m in MODELS}


paired = usable_accounts(r)
use = r[[a in paired[m] for m, a in zip(r["model"], r["account"], strict=True)]].copy()
for m in MODELS:
    print(f"{m}: accounts usable in both conditions {len(paired[m])}")

rows = []
for (model, condition, stratum), part in use.groupby(["model", "condition", "group"]):
    k, n = int(part["says"].sum()), len(part)
    low, high = wilson(k, n)
    rows.append([model, condition, stratum, k, n, k / n, low, high])
flagged = pd.DataFrame(
    rows,
    columns=["model", "condition", "stratum", "flagged", "n", "share", "low", "high"],
)
for model in MODELS:
    mine = flagged[flagged["model"] == model].copy()
    mine["text"] = mine.apply(
        lambda r: (
            f"{r['flagged']:>2}/{r['n']:<2} {r['share']:.2f} "
            f"[{r['low']:.2f}, {r['high']:.2f}]"
        ),
        axis=1,
    )
    print(model)
    print(
        mine.pivot(index="stratum", columns="condition", values="text")
        .reindex(ORDER)[CONDITIONS]
        .to_string()
    )
    print()

gemma4:latest: accounts usable in both conditions 255
qwen3.5:9b-q4_K_M: accounts usable in both conditions 255
gemma4:latest
condition                                        K1           K1_guide_shots
stratum                                                                     
pattern BIPARTITE           10/15 0.67 [0.42, 0.85]   4/15 0.27 [0.11, 0.52]
pattern CYCLE                9/15 0.60 [0.36, 0.80]   8/15 0.53 [0.30, 0.75]
pattern FAN-IN              10/15 0.67 [0.42, 0.85]   2/15 0.13 [0.04, 0.38]
pattern FAN-OUT              8/15 0.53 [0.30, 0.75]   3/15 0.20 [0.07, 0.45]
pattern GATHER-SCATTER       8/15 0.53 [0.30, 0.75]   3/15 0.20 [0.07, 0.45]
pattern RANDOM               9/15 0.60 [0.36, 0.80]   8/15 0.53 [0.30, 0.75]
pattern SCATTER-GATHER      10/15 0.67 [0.42, 0.85]   5/15 0.33 [0.15, 0.58]
pattern STACK                9/15 0.60 [0.36, 0.80]   4/15 0.27 [0.11, 0.52]
positive, outside the file   8/15 0.53 [0.30, 0.75]   0/15 0.00 [0.00, 0.20]
clean, matched activity    

## 3. Paired comparison against K1

The main result: the AUC-ROC change from K1 to the guided reading on the same accounts, with a 95% interval, and how many decisions became right or wrong.

In [4]:
def draws_for(strata, seed):
    rng = np.random.default_rng(seed)
    index = [np.flatnonzero(strata == s) for s in ORDER]
    return [
        np.concatenate(
            [rng.choice(ix, size=len(ix), replace=True) for ix in index if len(ix)]
        )
        for _ in range(DRAWS)
    ]


def paired_change(y, base, new, draws):
    a, b = roc_auc_score(y, base), roc_auc_score(y, new)
    boot = np.array(
        [roc_auc_score(y[d], new[d]) - roc_auc_score(y[d], base[d]) for d in draws]
    )
    low, high = np.percentile(boot, [2.5, 97.5])
    rule = "helped" if low > 0 else ("hurt" if high < 0 else "a help was not shown")
    return a, b, b - a, low, high, rule


def wide_for(frame, model, accounts):
    return (
        frame[frame["model"] == model]
        .pivot(index="account", columns="condition", values=["says", "conf_score"])
        .loc[accounts]
    )


mcnemar_rows, auc_rows = [], []
for model in MODELS:
    accounts = paired[model]
    y = truth["seed 102 pool"].loc[accounts].to_numpy()
    draws = draws_for(pool.loc[accounts].to_numpy(), SEED)
    wide = wide_for(use, model, accounts)
    right = {
        c: wide[("says", c)].astype(bool).to_numpy() == (y == 1) for c in CONDITIONS
    }
    for side, mask in [("positives", y == 1), ("clean", y == 0)]:
        better = int((~right[READ_ALONE] & right[CHOSEN] & mask).sum())
        worse = int((right[READ_ALONE] & ~right[CHOSEN] & mask).sum())
        p = binomtest(better, better + worse, 0.5).pvalue if better + worse else np.nan
        mcnemar_rows.append(
            [
                model,
                side,
                int(mask.sum()),
                int((right[READ_ALONE] & mask).sum()),
                int((right[CHOSEN] & mask).sum()),
                better,
                worse,
                p,
            ]
        )
    base = wide[("conf_score", READ_ALONE)].to_numpy(float)
    new = wide[("conf_score", CHOSEN)].to_numpy(float)
    auc_rows.append([model, *paired_change(y, base, new, draws)])

mcnemar = pd.DataFrame(
    mcnemar_rows,
    columns=[
        "model",
        "accounts",
        "n",
        "right K1",
        "right guided",
        "became right",
        "became wrong",
        "exact p",
    ],
)
auc = pd.DataFrame(
    auc_rows,
    columns=["model", "AUC K1", "AUC guided", "change", "low 95%", "high 95%", "rule"],
)
print(f"McNemar, decisions that change against K1 ({CHOSEN})")
print(mcnemar.round(4).to_string(index=False))
print()
print(f"AUC-ROC (confidence score), stratified paired bootstrap {DRAWS}")
print(auc.round(3).to_string(index=False))

McNemar, decisions that change against K1 (K1_guide_shots)
            model  accounts   n  right K1  right guided  became right  became wrong  exact p
    gemma4:latest positives 135        81            37             9            53      0.0
    gemma4:latest     clean 120        81           117            37             1      0.0
qwen3.5:9b-q4_K_M positives 135        97            50            14            61      0.0
qwen3.5:9b-q4_K_M     clean 120        52           117            65             0      0.0

AUC-ROC (confidence score), stratified paired bootstrap 2000
            model  AUC K1  AUC guided  change  low 95%  high 95%                 rule
    gemma4:latest   0.643       0.694   0.051   -0.020     0.117 a help was not shown
qwen3.5:9b-q4_K_M   0.565       0.672   0.106    0.022     0.183               helped


## 4. Every reader on the same accounts

The XGBoost transaction model and ceiling classifier beside the LLM readers, to show how far the LLMs are from a trained model.

In [5]:
readers = fresh.ml_readers(
    pool.index,
    pd.read_parquet(
        DATA_INTERIM / "transaction_reading.parquet", columns=["score", "flag"]
    ),
    pd.read_parquet(DATA_INTERIM / "ceiling_facts_scores.parquet"),
)
readers.update(fresh.llm_readers(r))
table = fresh.reader_table(pool, readers)
printed = pd.read_csv(pool_dir / "summary.csv")
assert np.allclose(
    table.drop(columns="reader").to_numpy(float),
    printed.drop(columns="reader").to_numpy(float),
)
print("same as the table printed by the run script")

lows, highs = [], []
for score, _ in readers.values():
    y = truth["seed 102 pool"].loc[score.index].to_numpy()
    s = score.to_numpy()
    boot = [
        roc_auc_score(y[d], s[d])
        for d in draws_for(pool.loc[score.index].to_numpy(), SEED)
    ]
    low, high = np.percentile(boot, [2.5, 97.5])
    lows.append(low)
    highs.append(high)
table["AUC low 95%"], table["AUC high 95%"] = lows, highs
print(table.round(3).to_string(index=False))

same as the table printed by the run script


                              reader  accounts  positives  AUC-ROC  recall  false alarm rate  AUC low 95%  AUC high 95%
           XGBoost transaction model       255        135    0.903   0.659             0.008        0.864         0.939
XGBoost ceiling (evidence and facts)       255        135    0.892   0.422             0.000        0.853         0.929
                           gemma4 K1       255        135    0.643   0.600             0.325        0.577         0.711
               gemma4 K1_guide_shots       255        135    0.694   0.274             0.025        0.638         0.748
                          qwen3.5 K1       255        135    0.565   0.719             0.567        0.503         0.627
              qwen3.5 K1_guide_shots       255        135    0.672   0.370             0.025        0.622         0.718


## 5. Four samples side by side

The same measures on the development, test, seed 101 and this sample, to see whether the change has a stable size.

In [6]:
rows = []
for name, frame in results.items():
    ok = usable_accounts(frame)
    for model in MODELS:
        accounts = ok[model]
        y = truth[name].loc[accounts].to_numpy()
        wide = wide_for(frame, model, accounts)
        aucs = {
            c: roc_auc_score(y, wide[("conf_score", c)].to_numpy(float))
            for c in CONDITIONS
        }
        for condition in CONDITIONS:
            says = wide[("says", condition)].astype(bool).to_numpy()
            rows.append(
                {
                    "sample": name,
                    "model": model.split(":")[0],
                    "condition": condition,
                    "accounts": len(accounts),
                    "AUC confidence": aucs[condition],
                    "change vs K1": aucs[condition] - aucs[READ_ALONE],
                    "recall (all positives)": says[y == 1].mean(),
                    "false alarm rate": says[y == 0].mean(),
                }
            )
side = pd.DataFrame(rows)
print(side.round(3).to_string(index=False))

        sample   model      condition  accounts  AUC confidence  change vs K1  recall (all positives)  false alarm rate
           dev  gemma4             K1       255           0.597         0.000                   0.600             0.433
           dev  gemma4 K1_guide_shots       255           0.692         0.095                   0.237             0.042
           dev qwen3.5             K1       255           0.536         0.000                   0.733             0.625
           dev qwen3.5 K1_guide_shots       255           0.685         0.149                   0.348             0.025
          test  gemma4             K1       255           0.637         0.000                   0.593             0.342
          test  gemma4 K1_guide_shots       255           0.755         0.118                   0.341             0.008
          test qwen3.5             K1       255           0.540         0.000                   0.704             0.592
          test qwen3.5 K1_guide_shots   

## 6. Results

- AUC-ROC change: Qwen +0.106 [+0.022; +0.183], Gemma +0.051 [-0.020; +0.117] (interval contains 0).
- Positive on all four samples; on this sample about half the test sample change (+0.118, +0.197).
- False alarms 39 and 68 of 120 to 3 and 3; recall 0.60 to 0.27 (Gemma), 0.72 to 0.37 (Qwen).
- Positives outside the pattern file flagged when guided: 0 of 15.
- XGBoost transaction model 0.903, ceiling 0.892: far above every LLM reader; here the transaction model is slightly above the ceiling.